# ACCESS Metrics report for Purdue

In [ ]:
PROVIDER = 'Purdue'
RESOURCE_RENAMES = {
    'Purdue Anvil CPU': 'Anvil CPU',
    'Purdue Anvil GPU': 'Anvil GPU',
}

In [ ]:
YEAR = 2025
QUARTER_START_DATE_SUFFIX = '-01-01'
QUARTER_START_DATE = str(YEAR) + QUARTER_START_DATE_SUFFIX
QUARTER_END_DATE_SUFFIX = '-03-31'
QUARTER_END_DATE = str(YEAR) + QUARTER_END_DATE_SUFFIX
TWO_YEARS_AGO_QUARTER_START_DATE = str(YEAR - 2) + QUARTER_START_DATE_SUFFIX

In [ ]:
# This cell will be removed once JWT implementation is complete.
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path(os.path.expanduser('~/xdmod-data.env')), override=True)
os.environ['XDMOD_API_TOKEN'] = os.environ['PROD_API_TOKEN']

In [ ]:
import sys
! {sys.executable} -m pip install --upgrade 'xdmod-data>=1.0.0,<2.0.0' python-dotenv tabulate
! {sys.executable} -m pip install  --upgrade xdmod-data
! {sys.executable} -m pip install pandas
! {sys.executable} -m pip show pandas
! {sys.executable} -m pip install itables

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import xdmod_data.themes
import itables
pio.templates.default = 'timeseries'
from xdmod_data.warehouse import DataWarehouse
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML, Javascript
try:
    pd.set_option('future.no_silent_downcasting',True)
except pd.errors.OptionError:
    pass
def display_df_md_table(df):
    return display(Markdown(df.replace('\n', '<br/>', regex=True).to_markdown(floatfmt=',.0f')))
dw = DataWarehouse('https://xdmod.access-ci.org')

In [ ]:
def create_plot(metric_label, resource, dimension_label, df,vertical_legend,nlargest):
        
        top_dimension_labels = None
        category_orders = None
        current_quarter_top_projects = []
        title = metric_label + (
            (f' on {resource}')
            if resource != 'all'
            else ''
        ) + f' by {dimension_label} by Quarter, Last Two Years'

        if nlargest > 0:
            
            list1 = df['Date'].unique().tolist()
            top_dimension_labels = []
            CURRENT_QUARTER = QUARTER_START_DATE + " 00:00:00"
                
            for date in list1:
                
                list2 = df[
                    df['Date'] == date
                ].nlargest(
                    nlargest,
                    metric_label,
                )[dimension_label].tolist()
                str_date = date.strftime('%Y-%m-%d %H:%M:%S')
    
                if str_date == CURRENT_QUARTER:
                    current_quarter_top_projects += list2
                    
                top_dimension_labels += list2
            top_dimension_labels = list(set(top_dimension_labels))
            df = df[df[dimension_label].isin(top_dimension_labels)]
            category_orders = {
                    dimension_label: top_dimension_labels,
                }
            title += f', Top {nlargest}'
            
            
            
        plot = px.line(
            df,
            x='Date',
            y=metric_label,
            title=title,
            color=dimension_label,
            markers=True,
            category_orders=category_orders,
        )
        plot.update_traces(
            hovertemplate='%{y:,.0f}',
        )
        plot.update_layout(
            xaxis_tickformat='Q%q %Y',
            hovermode='x unified',
            hoverlabel_namelength=-1,
        )
        if vertical_legend:
            plot.update_layout(
                legend_orientation='v',
                legend_xanchor='left',
                legend_x=0,
                legend_yanchor='bottom',
                legend_y=-1.3
            )
        plot.show()
        
        return (top_dimension_labels, current_quarter_top_projects)


def two_year_line_plot_by_quarter(
    y=None,
    resource=None,
    dimension=None,
    nlargest=0,
    vertical_legend=False
):
    if y == 'projects':
        metric = 'Number of Allocations: Active'
        metric_label = 'Number of Active Projects'
    elif y == 'users':
        metric = 'Number of Users: Active'
        metric_label = 'Number of Active Users'
    elif y == 'ace':
        metric = 'ACCESS Credit Equivalents Charged: Total (SU)'
        metric_label = 'ACCESS Credit Equivalents Charged'
    elif y == 'institutions':
        metric = 'Number of Institutions: Active'
        metric_label = 'Number of Institutions: Active'

    if resource == 'all':
        dimension = dimension_label = 'Resource'
        filters = {
            'Service Provider': PROVIDER,
        }
    else:
        filters = {
            'Resource': resource,
        }
    realm = 'Jobs'
    if dimension == 'pfos':
        dimension = 'Parent Science'
        dimension_label = 'Parent Field of Science'
    elif dimension == 'academic status':
        dimension = 'User NSF Status'
        dimension_label = 'User Academic Status'
    elif dimension == 'project':
        dimension = 'Allocation'
        dimension_label = 'Project'
    elif dimension == 'project type':
        realm = 'Allocations'
        dimension = 'Board Type'
        dimension_label = 'Project Type'
    with dw:
        df = dw.get_data(
            duration=(TWO_YEARS_AGO_QUARTER_START_DATE, QUARTER_END_DATE),
            realm=realm,
            metric=metric,
            dimension=dimension,
            dataset_type='timeseries',
            aggregation_unit='Quarter',
            filters=filters,
        )
    df = df.rename(
        columns={
            **RESOURCE_RENAMES,
            **{dimension: dimension_label}
        }
    )
    df = df.reset_index(names='Date')
    df = pd.melt(
        df,
        id_vars=['Date'],
        var_name=dimension_label,
        value_name=metric_label,
    
    )
    
    return create_plot(metric_label, resource, dimension_label, df,vertical_legend,nlargest)

## Active Projects

### Total

In [ ]:
two_year_line_plot_by_quarter(
    y='projects',
    resource='all',
)

### By Project Type

In [ ]:
def plot_num_projects_by_type(resource):
    with dw:
        df=dw.get_data(
            realm='Jobs',
            dimension = 'Allocation',
            filters = {
                'Resource': resource,
                },
            duration = (QUARTER_START_DATE, QUARTER_END_DATE),
            aggregation_unit='Quarter',
            metric = 'ACCESS Credit Equivalents Charged: Total (SU)'
        )

    list1 =df.columns.tolist()
    date_string_index = df.index.strftime('%Y-%m-%d')

    i=0
    while(i < 9):
        with dw:
           df2 =dw.get_raw_data(
               duration=(date_string_index[i],date_string_index[i+1]),
               realm='Allocations',
               fields = {'Name','Board Type'},
               filters = {'Resource': resource,}
            )
        filtered_df = df2[df2['Name'].isin(list1)].reset_index()
        filtered2_df = filtered_df.groupby(['Board Type']).count()
        filtered2_df = filtered2_df.reset_index()
        filtered2_df = filtered2_df.rename(columns={"Board Type": "Project Type", "Name": "Number of Active Projects"})
        filtered2_df = filtered2_df.drop(columns=['index'])
        filtered2_df.insert(0,"Date",date_string_index[i])
        if(i >= 1):
            filtered2_df = pd.concat([previous_df,filtered2_df])
        
        previous_df = filtered2_df
        i+=1
    create_plot(metric_label = 'Number of Active Projects', resource=resource, 
        dimension_label='Project Type', df=filtered2_df,vertical_legend = False,nlargest=0)


plot_num_projects_by_type('Purdue Anvil GPU')
plot_num_projects_by_type('Purdue Anvil CPU')

### Some Projects started as one type but later changed to a different type

### By Parent Field of Science

In [ ]:
two_year_line_plot_by_quarter(
    y='projects',
    resource='Purdue Anvil CPU',
    dimension='pfos',
)

two_year_line_plot_by_quarter(
    y='projects',
    resource='Purdue Anvil GPU',
    dimension='pfos',
)

### TITLEEE

In [ ]:
top_projects , current_quarter_top_projects = two_year_line_plot_by_quarter(
    y='ace',
    resource='Purdue Anvil CPU',
    dimension='project',
    nlargest=10,
    vertical_legend=True,
)


In [ ]:
resources = ['Purdue Anvil CPU', 'Purdue Anvil GPU']

with dw:
    for resource in resources:
        
        df_ace_charged = dw.get_data(
            duration = (QUARTER_START_DATE, QUARTER_END_DATE),
            realm = 'Jobs',
            metric = 'ACCESS Credit Equivalents Charged: Total (SU)',
            dimension = 'Allocation',
            dataset_type = 'aggregate',
            filters = {'Resource': resource},
        )
        
        df_jobs_ran = dw.get_data(
            duration = (QUARTER_START_DATE, QUARTER_END_DATE),
            realm = 'Jobs',
            metric = 'Number of Jobs Running',
            dimension ='Allocation',
            dataset_type ='aggregate',
            filters = {'Resource': resource},
        )
        
        df_all_projects = dw.get_raw_data(
                   duration = (QUARTER_START_DATE,QUARTER_END_DATE),
                   realm = 'Allocations',
                   filters = {'Resource': resource}
        )
        
        active_projects = df_ace_charged.index.tolist()
        
        
        df_active_projects = df_all_projects[df_all_projects['Name'].isin(active_projects)]
        df_active_projects = df_active_projects[['Name', 'Initial Start Date', 'Board Type','Field of Science','Parent Science']]
        df_active_projects = df_active_projects.rename(columns={
            'Board Type': 'Type',
            'Name': 'Title'
        })
        df_active_projects = df_active_projects.sort_values(by='Initial Start Date')
        df_active_projects = df_active_projects.groupby(
            ['Title','Field of Science','Parent Science']
        ).agg(
            {'Type': lambda x: ' -> '.join(list(dict.fromkeys(x)))}
        )
        df_active_projects = df_active_projects.reset_index().set_index('Title')
        
        df_active_projects = df_active_projects.join(df_ace_charged)
        df_active_projects = df_active_projects.join(df_jobs_ran)
        
        
        df_active_projects = df_active_projects.rename(columns={
            'Number of Jobs Running':'Number of Jobs',
            'ACCESS Credit Equivalents Charged: Total (SU)':'ACCESS Credit Equivalents Charged'
        })
        
        df_active_projects = df_active_projects.reset_index()
        
        
        df_active_projects[['Charge Number', 'Title']] = df_active_projects['Title'].str.split('-', n=1, expand=True)
        df_active_projects = df_active_projects.set_index('Charge Number')
        
        with pd.option_context("display.float_format", "{:,.2f}".format):
            itables.show(df_active_projects,column_filters= "header",columnDefs=[{"width": "500px","className": "dt-left", "targets": "_all",}],autoWidth=False)

### ALL PROJECTS

## Active Users

### Total

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='all',
)

### By Parent Field of Science

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil CPU',
    dimension='pfos',
)

two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil GPU',
    dimension='pfos',
)

### By Academic Status

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil CPU',
    dimension='academic status',
)

two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil GPU',
    dimension='academic status',
)

In [ ]:
with dw:
    metrics = dw.describe_metrics('Jobs')
display_df_md_table(metrics)

## Active User Institutions

### Total

In [ ]:
two_year_line_plot_by_quarter(
    y='institutions',
    resource='all',
    dimension=None,
)

In [ ]:
two_year_line_plot_by_quarter(
    y='ace',
    resource='all',
    dimension='pfos',
)

In [ ]:
resources = ['Purdue Anvil CPU', 'Purdue Anvil GPU']

with dw:
    for resource in resources:
        
        df_ace_charged = dw.get_data(
            duration = (QUARTER_START_DATE, QUARTER_END_DATE),
            realm = 'Jobs',
            metric = 'ACCESS Credit Equivalents Charged: Total (SU)',
            dimension = 'Allocation',
            dataset_type = 'aggregate',
            filters = {'Resource': resource},
        )
        
        df_jobs_ran = dw.get_data(
            duration = (QUARTER_START_DATE, QUARTER_END_DATE),
            realm = 'Jobs',
            metric = 'Number of Jobs Running',
            dimension ='Allocation',
            dataset_type ='aggregate',
            filters = {'Resource': resource},
        )
        
        df_all_projects = dw.get_raw_data(
                   duration = (QUARTER_START_DATE,QUARTER_END_DATE),
                   realm = 'Allocations',
                   filters = {'Resource': resource}
        )
        
        active_projects = df_ace_charged.index.tolist()
        
        
        df_active_projects = df_all_projects[df_all_projects['Name'].isin(active_projects)]
        df_active_projects = df_active_projects[['Name', 'Initial Start Date', 'Board Type','Field of Science','Parent Science']]
        df_active_projects = df_active_projects.rename(columns={
            'Board Type': 'Type',
            'Name': 'Title'
        })
        df_active_projects = df_active_projects.sort_values(by='Initial Start Date')
        
        df_active_projects = df_active_projects.groupby(
            ['Title','Field of Science','Parent Science']
        ).agg(
            {'Type': lambda x: ' -> '.join(list(dict.fromkeys(x)))
            }
        )
        
        
        df_active_projects = df_active_projects.reset_index().set_index('Title')
        
        df_active_projects = df_active_projects.join(df_ace_charged)
        df_active_projects = df_active_projects.join(df_jobs_ran)
        
        
        df_active_projects = df_active_projects.rename(columns={
            'Number of Jobs Running':'Number of Jobs',
            'ACCESS Credit Equivalents Charged: Total (SU)':'ACCESS Credit Equivalents Charged'
        })
        
        df_active_projects = df_active_projects.reset_index()
        
        
        df_active_projects[['Charge Number', 'Title']] = df_active_projects['Title'].str.split('-', n=1, expand=True)
        df_active_projects = df_active_projects.set_index('Charge Number')

        

        reorder_columns = ['ACCESS Credit Equivalents Charged',
                           'Number of Jobs',
                           'Type',
                           'Parent Science',
                           'Field of Science',
                           'Title'
                          ]
        df_active_projects = df_active_projects[reorder_columns]

        df_active_projects = df_active_projects.reset_index()

        title = 'ACCESS-Metrics-Report-' + resource.replace(' ', '-') + '-2025-Q1-Projects'

        css = """
        table.dataTable th.dt-type-numeric div.dt-column-header {
            flex-direction: row !important;
        }
        table.dataTable tfoot>tr>th div.dt-column-footer span.dt-column-title input {
            width: 100%
        }
        """
        display(HTML(f'<style>{css}</style>' ''))


        js_code = """
        $.fn.dataTable.ext.type.order['project-type-pre'] = function ( data ) {
  
            access_project_types = ['Explore', 'Discover', 'Accelerate', 'Maximize'];
            custom_sort_order = {};
            rank = 0;
            for (i = 0; i < access_project_types.length; i++) {
                custom_sort_order[access_project_types[i]] = rank++;
                if (i + 1 < access_project_types.length) {
                    custom_sort_order[access_project_types[i] + ' -> ' + access_project_types[i + 1]] = rank++;
                }
            }
          return custom_sort_order[data]; 
        };"""
        
        display(Javascript(js_code))




        

        itables.options.text_in_header_can_be_selected = False

        order = []

        itables.show(
            df_active_projects,
            buttons=[
                'pageLength', 'copyHtml5',
                {"extend": "csvHtml5", "title": title},
                {"extend": "excelHtml5", "title": title},
                "columnsToggle",
            ],
            column_filters= 'footer',
            order=[[1, 'desc']],
            autoWidth=False,
            allow_html=True,
            columnDefs=[{
            "targets": [1],
            "render": itables.JavascriptCode(
                "$.fn.dataTable.render.number(',' , '.' , '2')"
            ),},{"targets": [2],
                 "render": itables.JavascriptCode("$.fn.dataTable.render.number(',', '.',)")
                },{'width': '1px','className': 'dt-left','orderSequence': ['asc', 'desc'] , 'targets': '_all',},
                        {"className": "dt-body-right",'orderSequence': ['desc', 'asc'], "targets": [1, 2]},
                       ])


In [ ]:

format_str = '%Y-%m-%d'
day_before_two_years_ago_quarter_start_date = (datetime.strptime(TWO_YEARS_AGO_QUARTER_START_DATE, format_str).date() - timedelta(days=1)).strftime(format_str)


with dw:
    active_project1 = dw.get_data(
         realm = 'Jobs',
         duration = ('2015-01-01' , day_before_two_years_ago_quarter_start_date),
         metric = 'Number of Jobs Running', 
         dimension = 'Allocation',
         dataset_type = 'aggregate',
         aggregation_unit='quarter',
         filters = {'Resource': resource},
     )

    active_project2 = dw.get_data(
        realm = 'Jobs',
        duration = (TWO_YEARS_AGO_QUARTER_START_DATE , QUARTER_END_DATE),
        metric = 'Number of Jobs Running', 
        dimension = 'Allocation',
        dataset_type = 'timeseries',
        aggregation_unit='quarter',
        filters = {'Resource': resource},
    )
    
old_projects = active_project1.index.tolist()

active_project2 = active_project2.transpose()
active_project2 = active_project2.reset_index()

active_project2 = active_project2[~active_project2['Allocation'].isin(old_projects)]



num_per_quarter = []
list_of_quarters = active_project2.columns[1:].to_list()

for quarter in list_of_quarters:

    active_rows = active_project2[active_project2[quarter] > 0]
    num_per_quarter.append(len(active_rows))
    active_project2 = active_project2.drop(active_rows.index)


plot = px.line(
    x= list_of_quarters, 
    y= num_per_quarter ,
    
    title="Number of new active projects per quarter in the Last 2 years",
    
)
plot.update_traces(
    hovertemplate='%{y:,.0f}',
    )
plot.update_layout(
    xaxis_tickformat='Q%q %Y',
    hovermode='x unified',
    hoverlabel_namelength=-1,
    )


plot.update_xaxes(title_text='Last 2 years in Quarters')
plot.update_yaxes(title_text='Number of Active Projects')

plot.show()



In [ ]:
import plotly.graph_objects as go



df = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/2014_us_cities.csv')
df.head()



df['text'] = df['name'] + '<br>Population ' + (df['pop']/1e6).astype(str)+' million'

display(df)
limits = [(0,3),(3,11),(11,21),(21,50),(50,3000)]
colors = ["royalblue","crimson","lightseagreen","orange","lightgrey"]
cities = []
scale = 5000

fig = go.Figure()

for i in range(len(limits)):
    lim = limits[i]
    df_sub = df[lim[0]:lim[1]]
    fig.add_trace(go.Scattergeo(
        locationmode = 'USA-states',
        lon = df_sub['lon'],
        lat = df_sub['lat'],
        text = df_sub['text'],
        marker = dict(
            size = df_sub['pop']/scale,
            color = colors[i],
            line_color='rgb(40,40,40)',
            line_width=0.5,
            sizemode = 'area'
        ),
        name = '{0} - {1}'.format(lim[0],lim[1])))

fig.update_layout(
        title_text = '2014 US city populations<br>(Click legend to toggle traces)',
        showlegend = True,
        geo = dict(
            scope = 'usa',
            landcolor = 'rgb(217, 217, 217)',
        )
    )

fig.show()


In [ ]:
df = pd.read_csv('~/Downloads/xdmod-notebooks/user-inst-2025-q1.csv')
df.head()

df = df[df['total_ace'] != 0]

display_df_md_table(df)




In [ ]:


fig = go.Figure(data=go.Scattergeo(
    locationmode = 'USA-states',
    lon = df['longitude'],
    lat = df['latitude'],
    text = df['User Institution Name'],
    customdata=df['active_person_count'].values,
    hovertemplate="<b>User Institution</b>: %{text}<br><b>Number of Users Active</b>: %{customdata}<br><extra></extra>",
    mode = 'markers',
    marker = dict(
        size = df['active_person_count'], 
        color = 'blue',
        line_width = 0.5,
        sizemode = 'area'
    )))

fig.update_layout(
    title_text = 'User Institution in the U.S. by Active User count',
    showlegend = True,
    geo = dict(
        scope = 'usa',
        landcolor = 'rgb(217, 217, 217)'
    )
)
fig.update_layout(showlegend=False)

fig.show()

In [ ]:


fig = go.Figure(data=go.Scattergeo(
    locationmode = 'USA-states',
    lon = df['longitude'],
    lat = df['latitude'],
    text = df['User Institution Name'],
    customdata=df['total_ace'].values,
    hovertemplate="<b>User Institution</b>: %{text}<br><b>Total Ace</b>: %{customdata}<br><extra></extra>",
    mode = 'markers',
    marker = dict(
        size = df['total_ace']/100000, # had to divide by 100,000 or this would fill the whole map in red
        color = 'red',
        line_width = 0.5,
        sizemode = 'area'
    )))

fig.update_layout(
    title_text = 'User Institution in the U.S. by Total ACE',
    showlegend = True,
    geo = dict(
        scope = 'usa',
        landcolor = 'rgb(217, 217, 217)'
    )
)

fig.update_layout(showlegend=False)


fig.show()

In [ ]:
fig = go.Figure(data=go.Scattergeo(
    locationmode = 'USA-states',
    lon = df['longitude'],
    lat = df['latitude'],
    text = df['User Institution Name'],
    customdata=df['running_job_count'].values,
    hovertemplate="<b>User Institution</b>: %{text}<br><b>Running Job Count</b>: %{customdata}<br><extra></extra>",
    mode = 'markers',
    marker = dict(
        size = df['running_job_count'], 
        color = 'green',
        line_width = 0.5,
        sizemode = 'area'
    )))

fig.update_layout(
    title_text = 'User Institution in the U.S. by Running Job Count',
    showlegend = True,
    geo = dict(
        scope = 'usa',
        landcolor = 'rgb(217, 217, 217)'
    )
)

fig.update_layout(showlegend=False)

#fig.data[0].name = 'User Institution'
#jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace ACCESS-RP-Report.ipynb
fig.show()